# Day 17 — Recommended Order Quantity

Build a baseline order-quantity estimate from Day 16 outputs.

**Limitation:** Rossmann `Sales` is revenue, not physical units sold. Results are illustrative proxies, not real purchasing quantities. Inventory position is a placeholder until actual on-hand stock, open purchase orders, and backorders are available.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/processed")

input_path = DATA_DIR / "day16_safety_stock_reorder_point.csv"

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

inventory = pd.read_csv(input_path)

print("Loaded rows:", len(inventory))
print("Columns:", inventory.columns.tolist())
display(inventory.head())

Loaded rows: 1115
Columns: ['Store', 'Average_Daily_Sales', 'Daily_Sales_Std', 'Observed_Days', 'Lead_Time_Demand', 'Safety_Stock', 'Reorder_Point']


,Store,Average_Daily_Sales,Daily_Sales_Std,Observed_Days,Lead_Time_Demand,Safety_Stock,Reorder_Point
0,1,4759.096031,1012.106393,781,33313.672215,4404.951088,37718.623303
1,2,4953.900510,1610.149102,784,34677.303571,7007.788990,41685.092562
2,3,6942.568678,2193.383804,779,48597.980745,9546.178585,58144.159329
3,4,9638.401786,1936.031881,784,67468.812500,8426.115869,75894.928369
4,5,4676.274711,1765.745628,779,32733.922978,7684.985669,40418.908647


## Assumptions
Lead time and review period are illustrative. Replace inventory position with real data when available.

In [2]:
LEAD_TIME_DAYS = 7
REVIEW_PERIOD_DAYS = 7
ASSUMED_INVENTORY_POSITION = 0  # placeholder only

inventory["Lead_Time_Demand"] = inventory["Average_Daily_Sales"] * LEAD_TIME_DAYS
inventory["Review_Period_Demand"] = inventory["Average_Daily_Sales"] * REVIEW_PERIOD_DAYS
inventory["Target_Stock"] = inventory["Lead_Time_Demand"] + inventory["Review_Period_Demand"] + inventory["Safety_Stock"]
inventory["Assumed_Inventory_Position"] = ASSUMED_INVENTORY_POSITION
inventory["Recommended_Order_Quantity"] = (inventory["Target_Stock"] - inventory["Assumed_Inventory_Position"]).clip(lower=0)

display(inventory[["Store","Lead_Time_Demand","Review_Period_Demand","Safety_Stock","Reorder_Point","Target_Stock","Recommended_Order_Quantity"]].head(10))


,Store,Lead_Time_Demand,Review_Period_Demand,Safety_Stock,Reorder_Point,Target_Stock,Recommended_Order_Quantity
0,1,33313.672215,33313.672215,4404.951088,37718.623303,71032.295518,71032.295518
1,2,34677.303571,34677.303571,7007.788990,41685.092562,76362.396133,76362.396133
2,3,48597.980745,48597.980745,9546.178585,58144.159329,106742.140074,106742.140074
3,4,67468.812500,67468.812500,8426.115869,75894.928369,143363.740869,143363.740869
4,5,32733.922978,32733.922978,7684.985669,40418.908647,73152.831625,73152.831625
5,6,38569.623077,38569.623077,6865.169888,45434.792965,84004.416042,84004.416042
6,7,61719.356234,61719.356234,10657.028243,72376.384478,134095.740712,134095.740712
7,8,38775.508929,38775.508929,8243.593698,47019.102627,85794.611555,85794.611555
8,9,45936.363286,45936.363286,7088.326873,53024.690159,98961.053445,98961.053445
9,10,38978.946429,38978.946429,4574.804080,43553.750509,82532.696937,82532.696937


In [3]:
assert inventory["Store"].is_unique
assert inventory["Recommended_Order_Quantity"].notna().all()
assert (inventory["Recommended_Order_Quantity"] >= 0).all()
output_path = DATA_DIR / "day17_order_quantity.csv"
inventory.to_csv(output_path, index=False)
print(f"Saved {len(inventory)} rows to {output_path}")


Saved 1115 rows to ..\data\processed\day17_order_quantity.csv


## Conclusion
- Reused Day 16 safety-stock and reorder-point outputs.
- Estimated target stock using lead-time demand, review-period demand, and safety stock.
- Calculated baseline order quantity from target stock and assumed inventory position.
- Output: `data/processed/day17_order_quantity.csv`.
- Revenue-based proxy only; operational recommendations require unit demand, on-hand inventory, incoming orders, backorders, supplier lead times, and cost data.